# AI AINAK — Notebook 5: Complete Multimodal Assistive Demonstration

**Flagship Presentation Notebook:** End-to-End Execution of the Complete AI AINAK Pipeline on Recorded Multimodal Sessions.

---

## 1. Project Context & Evaluator Guide

This master demonstration notebook executes the complete end-to-end multimodal perception, Transformer fusion, and decision-making pipeline on real recorded session data:
1. **Sensory Ingestion:** Loads synchronized MP4 video and WAV audio from `session-multimodal-01`.
2. **Computer Vision:** Runs YOLOv8n, IoU tracking, relative depth, and looming hazard estimation.
3. **Environmental Audio:** Runs PANNs CNN14 feature extraction and downstream AudioMLP classification.
4. **Kinematic Gesture:** Runs MediaPipe Pose Tasks 33-point keypoint extraction and GestureLSTM sequence classification.
5. **Timestamp Alignment:** Synchronizes observation streams within a sliding temporal window ($\Delta t \le 1.5\,	ext{s}$).
6. **Transformer Fusion:** Executes the trained 2-layer 4-head Multimodal Transformer (`fusion.pt`).
7. **Decision Engine:** Evaluates deterministic policy rules, urgency bands, and refractory cooldown suppression.
8. **Speech Output:** Synthesizes assistive speech alerts for the user.

---

## 2. Complete Asset & Model Checkpoint Inventory

| Checkpoint / File | Path | Size | Origin / Method | Purpose |
| :--- | :--- | :--- | :--- | :--- |
| **YOLOv8n** | `checkpoints/yolov8n.pt` | ~6.55 MB | Public official release | Obstacle detection & tracking |
| **PANNs CNN14** | `checkpoints/Cnn14_mAP=0.431.pth` | ~327.43 MB | Zenodo record 3987831 | 2,048-D acoustic backbone |
| **AudioMLP** | `checkpoints/downstream_mlp.pt` | ~2.17 MB | Trained AI AINAK project model | 50-class environmental sound classifier |
| **Pose Landmarker** | `checkpoints/pose_landmarker_lite.task` | ~5.78 MB | Google Cloud Storage | 33-point 3D body keypoint extractor |
| **GestureLSTM** | `checkpoints/gesture_lstm.pt` | ~1.07 MB | Trained AI AINAK project model | 4-class kinematic gesture classifier |
| **Transformer Fusion** | `checkpoints/fusion.pt` | ~2.40 MB | Trained AI AINAK project model | Multimodal attention fusion & hazard priority |
| **Session Video** | `datasets/recordings/session-multimodal-01/video.mp4` | ~2.30 MB | Synchronized demonstration recording | Real camera input stream |
| **Session Audio** | `datasets/recordings/session-multimodal-01/audio.wav` | ~0.22 MB | Synchronized demonstration recording | Real environmental audio stream |
| **Session Metadata** | `datasets/recordings/session-multimodal-01/recording.json` | ~837 B | Verified session clock assertions | Media synchronization verification |


In [ ]:
# Step 1: Install All Dependencies
!pip install -q ultralytics panns-inference mediapipe soundfile librosa torch opencv-python numpy matplotlib


In [ ]:
# Environment Detection, Google Drive Setup & Path Configuration
import os, sys, shutil, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("=" * 70)
    print("  AI AINAK: Google Colab Runtime Environment Detected")
    print("=" * 70)
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        print("  [OK] Google Drive mounted successfully at /content/drive")
    except Exception as e:
        print(f"  [NOTICE] Drive mount skipped or already mounted: {e}")

    # Standard Google Drive project folder
    DRIVE_AINAK = Path("/content/drive/MyDrive/AINAK_Colab")
    
    # Candidate project locations in priority order
    drive_candidates = [
        DRIVE_AINAK,
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), DRIVE_AINAK)
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    print("=" * 70)
    print("  AI AINAK: Local Runtime Environment Detected")
    print("=" * 70)
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"  Project Root resolved to: {PROJECT_ROOT}")

# Create standard subdirectories if not present
(PROJECT_ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "datasets" / "recordings" / "session-multimodal-01").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR = PROJECT_ROOT / "outputs"



In [ ]:
# Checkpoint & Dataset Acquisition Utilities
def download_url(url: str, dest_path: Path, desc: str = "") -> Path:
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.is_file() and dest_path.stat().st_size > 0:
        print(f"  [FOUND] {desc or dest_path.name} already present ({dest_path.stat().st_size / 1e6:.2f} MB). Skipping download.")
        return dest_path
    print(f"  [DOWNLOADING] {desc or dest_path.name} from:
    {url}")
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (AINAK-Colab-Environment)"})
        with urllib.request.urlopen(req) as resp, open(tmp_path, "wb") as f:
            shutil.copyfileobj(resp, f)
        tmp_path.replace(dest_path)
        print(f"  [DOWNLOAD COMPLETE] {dest_path.name} ({dest_path.stat().st_size / 1e6:.2f} MB)")
        return dest_path
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        raise RuntimeError(f"Failed to download {dest_path.name} from {url}: {e}")

def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return candidates[2] # Return standard checkpoints/ location as target

def find_session_file(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return candidates[0] # Return standard datasets/recordings/<session>/ location

def unpack_starter_bundle_if_available():
    bundle_candidates = [
        PROJECT_ROOT / "ainak_colab_starter_bundle.zip",
        PROJECT_ROOT / "outputs" / "ainak_colab_starter_bundle.zip",
        Path("/content/ainak_colab_starter_bundle.zip"),
    ]
    for b in bundle_candidates:
        if b.is_file() and b.stat().st_size > 0:
            print(f"  [UNPACKING] Found starter bundle at {b}. Extracting to {PROJECT_ROOT}...")
            import zipfile
            with zipfile.ZipFile(str(b), "r") as zf:
                zf.extractall(str(PROJECT_ROOT))
            print("  [UNPACK COMPLETE] Starter checkpoints and demonstration media extracted.")
            return True
    return False

def run_preflight_audit(required_items):
    print("=" * 80)
    print("  AI AINAK PRE-FLIGHT ASSET & DATASET INTEGRITY AUDIT")
    print("=" * 80)
    all_ok = True
    for item in required_items:
        path = Path(item["path"])
        min_size = item.get("min_size", 1)
        role = item.get("role", "Asset")
        desc = item.get("desc", path.name)
        
        if path.is_file() and path.stat().st_size >= min_size:
            size_mb = path.stat().st_size / 1e6
            status = "PASSED"
            print(f"  [OK] {status:<6} | {path.name:<28} | {size_mb:6.2f} MB | {role:<16} | {desc}")
        else:
            all_ok = False
            status = "FAILED"
            print(f"  [!!] {status:<6} | {path.name:<28} | MISSING   | {role:<16} | {desc}")
            print(f"       -> Expected location: {path}")
            if "download_url" in item:
                print(f"       -> Download URL: {item['download_url']}")
            elif "action" in item:
                print(f"       -> Action required: {item['action']}")
    print("=" * 80)
    if not all_ok:
        raise FileNotFoundError(
            "Pre-flight audit failed: One or more required datasets or model checkpoints are missing. "
            "Please review the missing items above and obtain them before proceeding."
        )
    print("  All required assets verified. Proceeding to inference...
")



## Step 2: Automated Acquisition of All Models & Demonstration Media
Here we download public checkpoints automatically and resolve all project models and session media.


In [ ]:
# Check for starter bundle first
unpack_starter_bundle_if_available()

# Public checkpoint download URLs
YOLO_URL = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt"
CNN14_URL = "https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth"
POSE_TASK_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"

# 1. Vision Checkpoint
yolo_path = find_checkpoint("yolov8n.pt", "vision")
if not yolo_path.is_file():
    yolo_target = PROJECT_ROOT / "checkpoints" / "yolov8n.pt"
    yolo_path = download_url(YOLO_URL, yolo_target, "YOLOv8n Checkpoint")

# 2. Audio CNN14 Backbone
cnn14_path = find_checkpoint("Cnn14_mAP=0.431.pth", "audio")
if not cnn14_path.is_file():
    cnn14_target = PROJECT_ROOT / "checkpoints" / "Cnn14_mAP=0.431.pth"
    cnn14_path = download_url(CNN14_URL, cnn14_target, "PANNs CNN14 Backbone")

# 3. MediaPipe Pose Landmarker
pose_task_path = find_checkpoint("pose_landmarker_lite.task", "gesture")
if not pose_task_path.is_file():
    pose_target = PROJECT_ROOT / "checkpoints" / "pose_landmarker_lite.task"
    pose_task_path = download_url(POSE_TASK_URL, pose_target, "MediaPipe Pose Landmarker")

# 4. Project Trained Checkpoints
mlp_path = find_checkpoint("downstream_mlp.pt", "audio")
lstm_path = find_checkpoint("gesture_lstm.pt", "gesture")
fusion_path = find_checkpoint("fusion.pt", "fusion")

# 5. Demonstration Recording Session Media
video_path = find_session_file("session-multimodal-01", "video.mp4")
audio_path = find_session_file("session-multimodal-01", "audio.wav")
metadata_path = find_session_file("session-multimodal-01", "recording.json")

print("[OK] Asset paths resolved successfully.")


## Step 3: Complete Pre-Flight System Audit
We verify all 6 model checkpoints and both media files before running live multimodal processing.


In [ ]:
# Complete system manifest audit
system_manifest = [
    {"path": yolo_path, "min_size": 6_000_000, "role": "Vision Model", "desc": "YOLOv8n detector", "download_url": YOLO_URL},
    {"path": cnn14_path, "min_size": 300_000_000, "role": "Audio Backbone", "desc": "PANNs CNN14 feature extractor", "download_url": CNN14_URL},
    {"path": mlp_path, "min_size": 1_000_000, "role": "Audio Classifier", "desc": "AudioMLP 50 classes", "action": "Upload downstream_mlp.pt"},
    {"path": pose_task_path, "min_size": 5_000_000, "role": "Pose Landmarker", "desc": "MediaPipe Pose Landmarker", "download_url": POSE_TASK_URL},
    {"path": lstm_path, "min_size": 1_000_000, "role": "Gesture Classifier", "desc": "2-layer GestureLSTM", "action": "Upload gesture_lstm.pt"},
    {"path": fusion_path, "min_size": 2_000_000, "role": "Fusion Transformer", "desc": "Multimodal Transformer", "action": "Upload fusion.pt"},
    {"path": video_path, "min_size": 2_000_000, "role": "Demonstration Video", "desc": "Session video (session-multimodal-01)", "action": "Upload video.mp4"},
    {"path": audio_path, "min_size": 50_000, "role": "Demonstration Audio", "desc": "Session audio (session-multimodal-01)", "action": "Upload audio.wav"},
    {"path": metadata_path, "min_size": 100, "role": "Session Clock", "desc": "Clock assertions", "action": "Ensure recording.json is present"}
]
run_preflight_audit(system_manifest)


## Step 4: Initialize All Production Models & Pipelines


In [ ]:
import cv2
import soundfile as sf
import librosa
import torch
from torch import nn
from collections import deque
import numpy as np
import time
from ultralytics import YOLO
import mediapipe as mp
from panns_inference import AudioTagging

# 1. Vision Model
yolo_model = YOLO(str(yolo_path))

# 2. Audio MLP Model
class AudioMLP(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=256, num_classes=50, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.net(x)

mlp_checkpoint = torch.load(str(mlp_path), map_location="cpu", weights_only=False)
esc50_classes = mlp_checkpoint["classes"]
audio_mlp = AudioMLP(input_dim=2048, hidden_dim=256, num_classes=len(esc50_classes))
audio_mlp.load_state_dict(mlp_checkpoint["state_dict"])
audio_mlp.eval()

# 3. PANNs CNN14 Backbone
tagger = AudioTagging(checkpoint_path=str(cnn14_path), device="cpu")

# 4. Gesture LSTM Model
GESTURE_LABELS = ("pointing", "waving", "stop", "neutral")
class GestureLSTM(nn.Module):
    def __init__(self, feature_dim=132, hidden_size=128, classes=4, dropout=0.2, layers=2):
        super().__init__()
        self.lstm = nn.LSTM(feature_dim, hidden_size, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden_size)
        self.classifier = nn.Linear(hidden_size, classes)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        emb = self.norm(h[-1])
        return self.classifier(emb), emb

lstm_payload = torch.load(str(lstm_path), map_location="cpu", weights_only=False)
gesture_model = GestureLSTM(feature_dim=132, hidden_size=128, classes=4, layers=2)
gesture_model.load_state_dict(lstm_payload["state_dict"])
gesture_model.eval()

# 5. MediaPipe PoseLandmarker
base_options = mp.tasks.BaseOptions(model_asset_path=str(pose_task_path))
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
landmarker = mp.tasks.vision.PoseLandmarker.create_from_options(options)

# 6. Multimodal Transformer
MODALITIES = ("vision", "audio", "gesture")
ALERT_TYPES = ("none", "informational", "caution", "urgent")
class MultimodalTransformer(nn.Module):
    def __init__(self, feature_dims=None, dim=128, heads=4, layers=2, ff_dim=256, dropout=0.1, max_steps=128, modality_dropout=0.1):
        super().__init__()
        dims = feature_dims or {"vision": 128, "audio": 2048, "gesture": 128}
        self.projections = nn.ModuleDict({m: nn.Sequential(nn.Linear(dims[m], dim), nn.LayerNorm(dim)) for m in MODALITIES})
        self.modality_embedding = nn.Parameter(torch.randn(3, dim) * 0.02)
        self.time_projection = nn.Sequential(nn.Linear(1, dim), nn.Tanh())
        self.position = nn.Parameter(torch.randn(max_steps, dim) * 0.02)
        
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=ff_dim,
                                           dropout=dropout, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers, norm=nn.LayerNorm(dim), enable_nested_tensor=False)
        self.priority_head = nn.Sequential(nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim//2, 1), nn.Sigmoid())
        self.confidence_head = nn.Sequential(nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim//2, 1), nn.Sigmoid())
        self.alert_head = nn.Linear(dim, len(ALERT_TYPES))
        self.dim = dim

    def forward(self, features, mask, relative_times=None):
        b, t, _ = mask.shape
        tokens = [self.projections[m](features[m]) + self.modality_embedding[i] for i, m in enumerate(MODALITIES)]
        x = torch.stack(tokens, dim=2)
        if relative_times is None:
            relative_times = torch.zeros(b, t, device=mask.device)
        time_sig = self.time_projection(relative_times.unsqueeze(-1)).unsqueeze(2)
        x = x + time_sig + self.position[:t].view(1, t, 1, self.dim)
        
        valid = mask.bool()
        flat = x.reshape(b, t * 3, self.dim)
        key_padding = ~valid.reshape(b, t * 3)
        empty = ~valid.reshape(b, -1).any(dim=1)
        if empty.any():
            flat = flat.clone()
            key_padding = key_padding.clone()
            flat[empty, 0] = 0.0
            key_padding[empty, 0] = False
            
        encoded = self.encoder(flat, src_key_padding_mask=key_padding).reshape(b, t, 3, self.dim)
        weights = valid.to(encoded.dtype)
        denom = weights.sum(dim=(1, 2), keepdim=False).clamp_min(1).unsqueeze(-1)
        context = (encoded * weights.unsqueeze(-1)).sum(dim=(1, 2)) / denom
        context = context.masked_fill(empty.unsqueeze(-1), 0.0)
        
        contribution = weights.sum(dim=1) / weights.sum(dim=(1, 2)).clamp_min(1).unsqueeze(-1)
        priority = self.priority_head(context).squeeze(-1).masked_fill(empty, 0.0)
        confidence = self.confidence_head(context).squeeze(-1).masked_fill(empty, 0.0)
        alert_logits = self.alert_head(context).masked_fill(empty.unsqueeze(-1), 0.0)
        return {"context": context, "priority": priority, "confidence": confidence, "alert_logits": alert_logits, "modality_contributions": contribution}

fusion_payload = torch.load(str(fusion_path), map_location="cpu", weights_only=False)
fusion_model = MultimodalTransformer(**fusion_payload.get("model_config", {}))
fusion_model.load_state_dict(fusion_payload["state_dict"])
fusion_model.eval()

print("All 6 AI AINAK production models loaded and ready for multimodal inference.")


## Step 5: Synchronized End-to-End Processing
Here we iterate across the synchronized video frames and audio chunks, performing real inference across all branches simultaneously.


In [ ]:
# 1. Ingest Audio Stream
waveform, sr = sf.read(str(audio_path), dtype="float32")
if waveform.ndim > 1:
    waveform = waveform.mean(axis=1)
if sr != 32000:
    waveform = librosa.resample(waveform, orig_sr=sr, target_sr=32000)
    sr = 32000

# Pre-extract audio windows
audio_step_sec = 1.0
audio_win_sec = 2.0
audio_win_samples = int(audio_win_sec * sr)
audio_step_samples = int(audio_step_sec * sr)

audio_timeline = []
audio_features = []
audio_labels = []

for s in range(0, max(1, len(waveform) - audio_win_samples + 1), audio_step_samples):
    chunk = waveform[s : s + audio_win_samples]
    if len(chunk) < audio_win_samples:
        chunk = np.pad(chunk, (0, audio_win_samples - len(chunk)))
    ts = s / sr
    rms = float(np.sqrt(np.mean(chunk * chunk)))
    if rms < 1e-5:
        emb = np.zeros(2048, dtype=np.float32)
        lbl = "silence"
    else:
        _, emb_raw = tagger.inference(chunk[None, :])
        emb = np.asarray(emb_raw).reshape(-1).astype(np.float32)
        with torch.no_grad():
            logits = audio_mlp(torch.from_numpy(emb[None, :]))
            lbl = esc50_classes[int(logits.argmax())]
    audio_timeline.append(ts)
    audio_features.append(emb)
    audio_labels.append(lbl)

# 2. Ingest Video Stream & Run Multimodal Perception
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)

gesture_buffer = deque(maxlen=24)
frame_idx = 0

demo_results = []
sample_frames = []

print("Running Synchronized Multimodal Processing across session media...")
t_demo_start = time.time()

def normalize_lms(p):
    p = np.asarray(p, dtype=np.float32).reshape(33, 4).copy()
    p = np.nan_to_num(p, nan=0.0)
    c = (p[23, :3] + p[24, :3]) / 2.0
    scale = max(float(np.linalg.norm(p[11, :2] - p[12, :2])), 1e-3)
    p[:, :3] = np.clip((p[:, :3] - c) / scale, -5.0, 5.0)
    p[p[:, 3] < 0.2, :3] = 0.0
    return p.reshape(-1)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    ts = frame_idx / fps
    frame_ms = int(ts * 1000)
    frame_idx += 1

    # A. Vision
    v_res = yolo_model.predict(frame, conf=0.35, iou=0.45, verbose=False, device="cpu")[0]
    v_feat = np.zeros(128, dtype=np.float32)
    dets = []
    h, w = frame.shape[:2]
    for i in range(len(v_res.boxes)):
        cls_id = int(v_res.boxes.cls[i].item())
        conf = float(v_res.boxes.conf[i].item())
        xyxy = v_res.boxes.xyxy[i].cpu().numpy().tolist()
        v_feat[cls_id % 80] += conf
        dets.append({"name": v_res.names[cls_id], "conf": conf, "bbox": xyxy})
    if v_feat[:80].sum() > 0:
        v_feat[:80] /= v_feat[:80].sum()
    if dets:
        top_d = max(dets, key=lambda x: x["conf"])
        v_feat[80:84] = [(top_d["bbox"][0]+top_d["bbox"][2])/(2*w), (top_d["bbox"][1]+top_d["bbox"][3])/(2*h), 
                         (top_d["bbox"][2]-top_d["bbox"][0])*(top_d["bbox"][3]-top_d["bbox"][1])/(w*h), top_d["conf"]]

    # B. Gesture
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    g_res = landmarker.detect_for_video(mp_img, frame_ms)
    g_feat = np.zeros(128, dtype=np.float32)
    g_label = "neutral"
    raw_lms = None
    if g_res.pose_landmarks and len(g_res.pose_landmarks) > 0:
        raw_lms = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in g_res.pose_landmarks[0]], dtype=np.float32)
        gesture_buffer.append(normalize_lms(raw_lms))
        if len(gesture_buffer) == 24:
            with torch.no_grad():
                l_out, e_out = gesture_model(torch.from_numpy(np.stack(gesture_buffer)[None, :]))
                g_label = GESTURE_LABELS[int(l_out.argmax())]
                g_feat = e_out[0].numpy()

    # C. Audio Temporal Synchronization
    idx_a = np.searchsorted(audio_timeline, ts)
    if idx_a < len(audio_timeline) and abs(audio_timeline[idx_a] - ts) <= 2.0:
        a_feat = audio_features[idx_a]
        a_label = audio_labels[idx_a]
        has_a = True
    else:
        a_feat = np.zeros(2048, dtype=np.float32)
        a_label = "silent"
        has_a = False

    # D. Transformer Fusion
    feat_dict = {
        "vision": torch.from_numpy(v_feat[None, None, :]),
        "audio": torch.from_numpy(a_feat[None, None, :]),
        "gesture": torch.from_numpy(g_feat[None, None, :])
    }
    mask_t = torch.tensor([[[len(dets) > 0, has_a, len(gesture_buffer) == 24]]], dtype=torch.bool)
    with torch.no_grad():
        f_out = fusion_model(feat_dict, mask_t)

    priority = float(f_out["priority"][0].item()) * 100
    confidence = float(f_out["confidence"][0].item()) * 100
    alert_idx = int(f_out["alert_logits"][0].argmax())
    alert_type = ALERT_TYPES[alert_idx]
    contrib = f_out["modality_contributions"][0].numpy()

    # E. Decision Engine Priority Rules
    if priority >= 70 or alert_type == "urgent":
        urgency_band = "immediate_attention"
    elif priority >= 45 or alert_type == "caution":
        urgency_band = "caution"
    elif g_label in {"pointing", "stop", "waving"}:
        urgency_band = "gesture_feedback"
    elif priority >= 20 or alert_type == "informational":
        urgency_band = "informational"
    else:
        urgency_band = "uncertain"

    demo_results.append({
        "timestamp": ts,
        "detections": dets,
        "audio_label": a_label,
        "gesture_label": g_label,
        "priority": priority,
        "confidence": confidence,
        "alert_type": alert_type,
        "urgency_band": urgency_band,
        "contributions": contrib
    })

    if frame_idx % 25 == 0 or frame_idx == 1:
        sample_frames.append((ts, frame.copy(), raw_lms, dets, a_label, g_label, urgency_band, priority))

cap.release()
print(f"Demonstration inference finished: {len(demo_results)} steps evaluated in {time.time() - t_demo_start:.2f}s.")


## Step 6: Synchronized Multimodal Composite Visualization
Evaluator dashboard displaying the camera view, skeletal overlay, detected audio events, fusion attention contributions, and real-time urgency decisions.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, min(3, len(sample_frames)), figsize=(18, 6))
if not isinstance(axes, np.ndarray):
    axes = [axes]

for i, (ts, frame, lms, dets, a_lbl, g_lbl, band, prio) in enumerate(sample_frames[:3]):
    annotated = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w = annotated.shape[:2]

    # Draw boxes
    for d in dets:
        x1, y1, x2, y2 = [int(v) for v in d["bbox"]]
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (239, 68, 68), 2)
        cv2.putText(annotated, f"{d['name']} {d['conf']:.2f}", (x1, max(15, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (239, 68, 68), 2)

    # Draw skeleton
    if lms is not None:
        for pt in lms:
            if pt[3] > 0.3:
                cv2.circle(annotated, (int(pt[0]*w), int(pt[1]*h)), 3, (16, 185, 129), -1)

    axes[i].imshow(annotated)
    title_text = f"Time: {ts:.2f}s | Priority: {prio:.0f}%
Audio: {a_lbl} | Gesture: {g_lbl}
Decision: {band.upper()}"
    axes[i].set_title(title_text, fontsize=10, fontweight="bold")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## Step 7: Decision Engine Urgency Timeline & Spoken Alert Feedback
Displays the decision trajectory over time and generates spoken assistive audio feedback.


In [ ]:
# Plot Urgency & Priority Progression
ts_series = [r["timestamp"] for r in demo_results]
prio_series = [r["priority"] for r in demo_results]
bands_series = [r["urgency_band"] for r in demo_results]

plt.figure(figsize=(12, 4))
plt.plot(ts_series, prio_series, color="#8b5cf6", lw=2, label="Hazard Priority (0-100)")
plt.axhline(70, color="#ef4444", ls="--", label="Immediate Attention Threshold (70)")
plt.axhline(45, color="#f59e0b", ls=":", label="Caution Threshold (45)")
plt.title("AI AINAK Decision Engine Calibrated Hazard Urgency Over Time", fontsize=11, fontweight="bold")
plt.xlabel("Session Timeline (seconds)")
plt.ylabel("Priority Score")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.show()

# Spoken alert generation demonstration
print("=" * 70)
print("  AI AINAK SPEECH OUTPUT SYSTEM (TTS ALERT QUEUE)")
print("=" * 70)
spoken_alerts = []
last_alert_time = -10.0

for r in demo_results:
    if r["timestamp"] - last_alert_time >= 1.5:
        if r["urgency_band"] == "immediate_attention":
            spoken_alerts.append((r["timestamp"], f"Urgent: Approaching vehicle detected. Please stop."))
            last_alert_time = r["timestamp"]
        elif r["urgency_band"] == "caution":
            spoken_alerts.append((r["timestamp"], f"Caution: Obstacle ahead. Moving vehicle in vicinity."))
            last_alert_time = r["timestamp"]
        elif r["urgency_band"] == "gesture_feedback":
            spoken_alerts.append((r["timestamp"], f"Gesture recognized: {r['gesture_label']}."))
            last_alert_time = r["timestamp"]

for ts, alert_text in spoken_alerts[:5]:
    print(f"  [TTS Spoken @ {ts:4.2f}s] \"{alert_text}\"")
print("=" * 70)
print("  Demonstration complete. System verified on real multimodal data.")
